In [1]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
import torch

In [2]:
unknown_token = "<unk>"
pad_token = "<pad>"
eos_token = "<eos>"

dataset_folder = "wikitext-103/"
save_folder = "save-data/"

tokenizer = Tokenizer(BPE(unk_token=unknown_token, dropout=0.1))
# We need a pre-tokenizer to speed up training - without one, the model has to study the entire dataset in one go!
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
trainer = BpeTrainer(special_tokens=[unknown_token, pad_token, eos_token], vocab_size=35000, min_frequency=2, initial_alphabet=ByteLevel.alphabet())

In [3]:
# Train the tokenizer and save it
train_files = [dataset_folder+"wiki.train.raw"]
tokenizer.train(train_files, trainer)
tokenizer_save_location = save_folder+"wikitext-tokenizer.json"
tokenizer.save(tokenizer_save_location)
print(f"Tokenizer trained and saved to {tokenizer_save_location}")

Tokenizer trained and saved to save-data/wikitext-tokenizer.json


In [4]:
# Algorithm to split up each Wikipedia article, so the model can distinguish between different articles
# We have to stream the dataset content, or my PC will not survive the training dataset...
def preprocess_and_save_dataset(dataset_file, tokenizer, token_file):
    print(f"Processing dataset {dataset_file}...")
    
    dataset_ids = []
    current_article = []
    article_count = 0
    eos_id = tokenizer.token_to_id(eos_token)

    # We do not want to read the first line - it's just a blank line that will mess up our article counting logic
    first_line_read = False

    with open(dataset_file, 'r', encoding='utf-8') as f:
        for line in f:
            # We don't want the first line - skip it!
            if not first_line_read:
                first_line_read = True
                continue
            
            # If this is the start of a new article, save the old article
            # Use a stripped version of the line to sidestep any whitespace shenanigans
            clean_line = line.strip()
            if clean_line.startswith("= ") and clean_line.endswith(" =") and len(clean_line) > 2 and clean_line[2] != '=' and current_article:
                # This is the start of a new article, so we need to tokenize the previous article!
                previous_article_ids = tokenizer.encode("".join(current_article)).ids
                dataset_ids.extend(previous_article_ids)
                dataset_ids.append(eos_id)
                current_article = []
                article_count = article_count + 1
            # Add to the current article
            current_article.append(line)

        if current_article:
            # Add a tokenized version of the final article
            article_ids = tokenizer.encode("".join(current_article)).ids
            dataset_ids.extend(article_ids)
            dataset_ids.append(eos_id)
            article_count = article_count + 1

    # Save the tokenized articles
    print(f"Detected {article_count} articles in {dataset_file}")
    torch.save(torch.tensor(dataset_ids, dtype=torch.long), token_file)

In [5]:
# Tokenize and save every dataset
dataset_files = [dataset_folder+"wiki.train.raw", dataset_folder+"wiki.test.raw", dataset_folder+"wiki.valid.raw", dataset_folder+"wiki.train.reduced.raw"]
token_files = [save_folder+"train-dataset-tokens.pt", save_folder+"test-dataset-tokens.pt", save_folder+"valid-dataset-tokens.pt", save_folder+"train-reduced-dataset-tokens.pt"]

for dataset_file, token_file in zip(dataset_files, token_files):
    preprocessed_dataset = preprocess_and_save_dataset(dataset_file, tokenizer, token_file)
    print(f"{dataset_file} saved to {token_file}\n")

Processing dataset wikitext-103/wiki.train.raw...
Detected 29444 articles in wikitext-103/wiki.train.raw
wikitext-103/wiki.train.raw saved to save-data/train-dataset-tokens.pt

Processing dataset wikitext-103/wiki.test.raw...
Detected 62 articles in wikitext-103/wiki.test.raw
wikitext-103/wiki.test.raw saved to save-data/test-dataset-tokens.pt

Processing dataset wikitext-103/wiki.valid.raw...
Detected 60 articles in wikitext-103/wiki.valid.raw
wikitext-103/wiki.valid.raw saved to save-data/valid-dataset-tokens.pt

Processing dataset wikitext-103/wiki.train.reduced.raw...
Detected 205 articles in wikitext-103/wiki.train.reduced.raw
wikitext-103/wiki.train.reduced.raw saved to save-data/train-reduced-dataset-tokens.pt

